# ConvNeXt: same-fold fine-tuning at 320 px

Target: improve the 0.94275 public-score baseline while keeping `convnext_tiny.fb_in1k`.
This experiment does not guarantee 0.97. No external data, new labels or new backbone.

Each fold restores ONLY its matching 224-px checkpoint from `convnext_cv`. Those models
originally used `pretrained=True` with ImageNet-1k weights. `pretrained=False` here avoids
re-downloading initialization because the entire trained state is restored before use.
Source checkpoint hashes, original split fingerprint and held-out IDs are verified.

Changes: 320 x 320 full-frame resize, six fine-tuning epochs, smaller backbone LR,
larger classifier LR, one-epoch step-wise warmup followed by cosine decay, and EMA weights.
Batch size 16 with two-step gradient accumulation keeps the effective batch near 32.
The augmentations, sqrt positive weights, labels, grouped folds and validation function
remain the same. We evaluate raw and EMA models at threshold 0.5 and save the better
candidate. Epoch 0 (the source weights evaluated at 320) is also a checkpoint candidate.

Start from the saved baseline Output. Set SOURCE_CV_DIR to `convnext_cv` (unzip the backup
first if necessary). Outputs go to a NEW `convnext_cv_320` directory. `last.pt` includes
raw weights, EMA weights, optimizer, scheduler and scaler for epoch-boundary continuation.
For a new session, set PREVIOUS_CV_DIR to saved `convnext_cv_320` output, not to the 224 run.

The first diagnostic table uses existing OOF probabilities and requires no GPU. The
fine-tuning cell then evaluates epoch 0 before training. Set FOLDS_TO_TRAIN = [0] for
a pilot, or keep all five for the full experiment. After all folds finish, run
`cnn_2_ensemble_320.ipynb`, then optionally `cnn_2_blend_224_320.ipynb` (no GPU).

Training runs use the shared W&B project and common val_f1 / val_loss names, with extra
raw/EMA curves. Keep the baseline and all new outputs separate. A 4–6 hour budget is an
estimate to test on your GPU, not a runtime guarantee; early stopping may shorten it.


In [1]:
# Optional, separate install cell; restart the kernel afterwards.
# %pip install -q timm wandb "albumentations>=2,<3"

In [2]:
import json
import os
import random
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

print('torch:', torch.__version__, '| timm:', timm.__version__)
import hashlib
import shutil
from time import perf_counter
import copy
import math

torch: 2.11.0+cu128 | timm: 1.0.29


## 1. Fine-tuning configuration

Default: all five folds, up to six epochs each. Only resume the new phase from its own last.pt. The source 224 checkpoints are read-only.

In [3]:
SEED = 21
MODEL_NAME = 'convnext_tiny.fb_in1k'
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
FOLDS_PATH = None
CV_DIR = Path('/kaggle/working/convnext_cv_320')
SOURCE_CV_DIR = Path('/kaggle/working/convnext_cv')
PREVIOUS_CV_DIR = None  # saved convnext_cv_320 from an earlier session
FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
IMAGE_SIZE = 320
BATCH_SIZE = 16
ACCUMULATION_STEPS = 2
EPOCHS = 6
EARLY_STOPPING_PATIENCE = 4
LR = 1e-5
HEAD_LR = 5e-5
WARMUP_EPOCHS = 1
EMA_DECAY = 0.99
MIN_LR = 2e-7
WEIGHT_DECAY = 1e-2
THRESHOLD = 0.5
NUM_WORKERS = 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
USE_WANDB = True
WANDB_PROJECT = 'cats-and-dogs'
WANDB_ENTITY = 'yrachynska-kyiv-school-of-economics'  # shared team project
WANDB_MODE = 'online'
WANDB_LOG_EVERY = 20
assert len(FOLDS_TO_TRAIN) == len(set(FOLDS_TO_TRAIN)) and set(FOLDS_TO_TRAIN).issubset(range(5))
assert EPOCHS > 0 and BATCH_SIZE > 0 and EARLY_STOPPING_PATIENCE > 0
assert ACCUMULATION_STEPS > 0 and 0 <= EMA_DECAY < 1 and 0 <= WARMUP_EPOCHS < EPOCHS
assert HEAD_LR >= LR > 0 and IMAGE_SIZE % 32 == 0
assert 0 <= MIN_LR <= LR and NUM_WORKERS == 0
assert CV_DIR.resolve() != SOURCE_CV_DIR.resolve(), 'Source and destination must differ.'
CV_DIR.mkdir(parents=True, exist_ok=True)
preprocessing = {'name': 'albumentations_resize_v1', 'image_size': IMAGE_SIZE,
                 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'interpolation': 1}
print('device:', DEVICE, '| requested folds:', FOLDS_TO_TRAIN, '| outputs:', CV_DIR)

device: cuda | requested folds: [0, 1, 2, 3, 4] | outputs: /kaggle/working/convnext_cv_320


In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

## 2. Load the original split

Validate IDs, labels, five-fold assignments and group separation. Fingerprint the exact CSV contents to reject stale checkpoints or changed splits.

In [5]:
def digest_text(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

def file_digest(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

def save_json(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(path)

def save_torch(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


if not (SOURCE_CV_DIR / 'ensemble_config.json').is_file():
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        if {'ensemble_config.json', 'cv_config.json', 'folds.csv', 'oof_predictions.csv'}.issubset(files):
            cfg = json.loads((Path(directory) / 'cv_config.json').read_text())
            if cfg.get('pipeline') == 'convnext_cv_v1' and cfg['preprocessing']['image_size'] == 224:
                candidates.append(Path(directory))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach the complete original convnext_cv output or set SOURCE_CV_DIR. Candidates: {candidates}')
    SOURCE_CV_DIR = candidates[0]
assert CV_DIR.resolve() != SOURCE_CV_DIR.resolve()
source_manifest = json.loads((SOURCE_CV_DIR / 'ensemble_config.json').read_text())
assert source_manifest['model_name'] == MODEL_NAME and source_manifest['label_cols'] == LABEL_COLS
assert source_manifest['folds'] == list(range(5))
assert source_manifest['preprocessing']['image_size'] == 224
for key in ['name', 'mean', 'std', 'interpolation']:
    assert source_manifest['preprocessing'][key] == preprocessing[key]
for f in range(5):
    assert file_digest(SOURCE_CV_DIR / f'fold_{f}' / 'best.pt') == source_manifest['checkpoint_hashes'][str(f)]
if FOLDS_PATH is None:
    FOLDS_PATH = SOURCE_CV_DIR / 'folds.csv'

if not ((DATA_DIR / 'train.csv').is_file() and (DATA_DIR / 'test.csv').is_file()):
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        root = Path(directory)
        if {'train.csv', 'test.csv'}.issubset(files) and (root / 'train').is_dir() and (root / 'test').is_dir():
            if {'image_id', *LABEL_COLS}.issubset(pd.read_csv(root / 'train.csv', nrows=0).columns):
                candidates.append(root)
    if len(candidates) != 1:
        raise FileNotFoundError(f'Set DATA_DIR to the competition data. Candidates: {candidates}')
    DATA_DIR = candidates[0]
if FOLDS_PATH is None:
    paths = [Path('/kaggle/working/folds.csv'), Path('folds.csv'), CV_DIR / 'folds.csv']
    if PREVIOUS_CV_DIR is not None:
        paths.append(Path(PREVIOUS_CV_DIR) / 'folds.csv')
    candidates = {p.resolve() for p in paths if p.is_file()}
    if not candidates:
        for directory, subdirs, files in os.walk('/kaggle/input'):
            subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
            if 'folds.csv' in files:
                candidates.add((Path(directory) / 'folds.csv').resolve())
    # Copies with identical contents represent the same split.
    by_contents = {}
    for path in sorted(candidates):
        by_contents.setdefault(file_digest(path), path)
    if len(by_contents) != 1:
        raise FileNotFoundError(f'Attach the ORIGINAL folds.csv or set FOLDS_PATH. No split is generated. Candidates: {sorted(candidates)}')
    FOLDS_PATH = next(iter(by_contents.values()))
FOLDS_PATH = Path(FOLDS_PATH)
train = pd.read_csv(DATA_DIR / 'train.csv', dtype={'image_id': str})
folds = pd.read_csv(FOLDS_PATH, dtype={'image_id': str})
assert {'image_id', 'cluster', 'fold', *LABEL_COLS}.issubset(folds.columns)
assert train.image_id.notna().all() and train.image_id.is_unique
assert folds.image_id.notna().all() and folds.image_id.is_unique
assert set(train.image_id) == set(folds.image_id)
assert train[LABEL_COLS].isin([0, 1]).all().all()
aligned = folds.set_index('image_id').loc[train.image_id]
assert np.array_equal(train[LABEL_COLS].to_numpy(), aligned[LABEL_COLS].to_numpy())
assert folds[['cluster', 'fold']].notna().all().all()
assert set(folds.fold.unique()) == set(range(5))
assert folds.groupby('cluster').fold.nunique().eq(1).all(), 'A group crosses folds.'
frame = train.merge(folds[['image_id', 'cluster', 'fold']], on='image_id', validate='one_to_one', sort=False)
frame['path'] = frame.image_id.map(lambda ident: str(DATA_DIR / 'train' / f'{ident}.jpg'))
canonical = folds[['image_id', *LABEL_COLS, 'cluster', 'fold']].sort_values('image_id').copy()
canonical[['cluster', 'fold', *LABEL_COLS]] = canonical[['cluster', 'fold', *LABEL_COLS]].astype('int64')
folds_fingerprint = digest_text(canonical.to_csv(index=False, lineterminator=chr(10)))
assert folds_fingerprint == source_manifest['folds_fingerprint'], 'Source and target splits differ.'
cv_config = {
    'pipeline': 'convnext_cv_320_ema_v1', 'model_name': MODEL_NAME, 'label_cols': LABEL_COLS,
    'versions': {'torch': str(torch.__version__), 'timm': str(timm.__version__), 'albumentations': str(A.__version__)},
    'source_cv_fingerprint': source_manifest['cv_fingerprint'],
    'source_checkpoint_hashes': source_manifest['checkpoint_hashes'],
    'head_lr': HEAD_LR, 'accumulation_steps': ACCUMULATION_STEPS,
    'ema_decay': EMA_DECAY, 'warmup_epochs': WARMUP_EPOCHS,
    'scheduler': 'per_optimizer_step_warmup_cosine',
    'folds_fingerprint': folds_fingerprint, 'seed': SEED, 'epochs': EPOCHS,
    'batch_size': BATCH_SIZE, 'lr': LR, 'min_lr': MIN_LR, 'weight_decay': WEIGHT_DECAY,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE, 'preprocessing': preprocessing,
    'train_loss': 'BCE_sqrt_pos_weight', 'val_loss': 'BCE_unweighted', 'threshold': THRESHOLD,
}
cv_fingerprint = digest_text(json.dumps(cv_config, sort_keys=True))
cv_config['cv_fingerprint'] = cv_fingerprint
config_path = CV_DIR / 'cv_config.json'
if config_path.is_file():
    assert json.loads(config_path.read_text()) == cv_config, 'CV settings changed; use a new CV_DIR.'
if PREVIOUS_CV_DIR is not None:
    PREVIOUS_CV_DIR = Path(PREVIOUS_CV_DIR)
    previous_config = json.loads((PREVIOUS_CV_DIR / 'cv_config.json').read_text())
    assert previous_config == cv_config, 'Previous CV output uses different settings or folds.'
    # Carry all saved folds forward even when training only one requested fold now.
    for previous_f in range(5):
        source_folder = PREVIOUS_CV_DIR / f'fold_{previous_f}'
        destination = CV_DIR / f'fold_{previous_f}'
        if not any((destination / name).exists() for name in ['best.pt', 'last.pt', 'complete.json']):
            destination.mkdir(parents=True, exist_ok=True)
            for name in ['best.pt', 'last.pt', 'history.csv', 'oof.csv', 'complete.json', 'wandb_id.txt', 'comparison.json']:
                if (source_folder / name).is_file():
                    shutil.copy2(source_folder / name, destination / name)

save_json(cv_config, config_path)
folds.to_csv(CV_DIR / 'folds.csv', index=False)
print('Original folds:', FOLDS_PATH, '| fingerprint:', folds_fingerprint[:12])
display(frame.groupby('fold')[LABEL_COLS].sum())

Original folds: /kaggle/working/convnext_cv/folds.csv | fingerprint: 453ed1d2e114


,Cat,Dog,Moris,Motya,Biatrix
fold,,,,,
0,1006,982,14,7,9
1,1027,1007,12,7,7
2,1024,1002,12,14,5
3,1031,1003,7,4,14
4,989,1031,11,10,6


## Source OOF diagnostics (no training)

Report precision, recall, FP/FN and F1 for each label using the saved 224 thresholds.
Save the error rows for inspection; do not manually relabel competition images.
Rare-label counts are small, so a few mistakes can move macro F1 substantially.


In [6]:
source_oof = pd.read_csv(SOURCE_CV_DIR / 'oof_predictions.csv', dtype={'image_id': str})
assert source_oof.image_id.is_unique and set(source_oof.image_id) == set(frame.image_id)
source_oof = source_oof.set_index('image_id').loc[frame.image_id].reset_index()
assert np.array_equal(source_oof[LABEL_COLS].to_numpy(), frame[LABEL_COLS].to_numpy())
assert np.array_equal(source_oof['fold'].to_numpy(), frame['fold'].to_numpy())
source_targets = source_oof[LABEL_COLS].to_numpy(dtype=np.int32)
audit_rows, errors = [], []
for method, suffix in [('original', '_prob'), ('tta', '_prob_tta')]:
    probabilities = source_oof[[label + suffix for label in LABEL_COLS]].to_numpy()
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    thresholds = np.asarray(source_manifest['thresholds'][method], dtype=np.float64)
    assert thresholds.shape == (len(LABEL_COLS),) and np.isfinite(thresholds).all()
    assert ((thresholds > 0) & (thresholds < 1)).all()
    for j, label in enumerate(LABEL_COLS):
        y = source_targets[:, j]
        predicted = probabilities[:, j] >= thresholds[j]
        fp, fn = predicted & (y == 0), (~predicted) & (y == 1)
        audit_rows.append({'method': method, 'label': label, 'positives': int(y.sum()),
                           'threshold': float(thresholds[j]), 'false_positives': int(fp.sum()),
                           'false_negatives': int(fn.sum()),
                           'precision': precision_score(y, predicted, zero_division=0),
                           'recall': recall_score(y, predicted, zero_division=0),
                           'f1': f1_score(y, predicted, zero_division=0)})
        for index in np.flatnonzero(fp | fn):
            errors.append({'image_id': source_oof.image_id.iloc[index], 'fold': int(source_oof.fold.iloc[index]),
                           'method': method, 'label': label, 'target': int(y[index]),
                           'probability': float(probabilities[index, j]), 'threshold': float(thresholds[j]),
                           'error': 'FP' if fp[index] else 'FN'})
audit = pd.DataFrame(audit_rows)
display(audit)
audit.to_csv(CV_DIR / 'source_224_per_label.csv', index=False)
pd.DataFrame(errors, columns=['image_id', 'fold', 'method', 'label', 'target', 'probability', 'threshold', 'error']).to_csv(CV_DIR / 'source_224_errors.csv', index=False)

,method,label,positives,threshold,false_positives,false_negatives,precision,recall,f1
0,original,Cat,5077,0.39,23,29,0.995464,0.994288,0.994876
1,original,Dog,5025,0.39,35,34,0.993036,0.993234,0.993135
2,original,Moris,56,0.62,1,3,0.981481,0.946429,0.963636
3,original,Motya,42,0.50,2,1,0.953488,0.976190,0.964706
4,original,Biatrix,41,0.50,0,2,1.000000,0.951220,0.975000
5,tta,Cat,5077,0.46,20,28,0.996054,0.994485,0.995269
6,tta,Dog,5025,0.49,33,34,0.993432,0.993234,0.993333
7,tta,Moris,56,0.50,2,3,0.963636,0.946429,0.954955
8,tta,Motya,42,0.59,2,2,0.952381,0.952381,0.952381
9,tta,Biatrix,41,0.50,0,2,1.000000,0.951220,0.975000


## 3. Fine-tuning helpers

Reuse the existing transforms and validation function. Gradient accumulation weights
microbatches by their actual image count, including a smaller final batch. Update EMA
and the warmup/cosine schedule only after an optimizer update (not after each microbatch).


In [7]:
def make_transforms():
    train_transform = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=preprocessing['interpolation']),
        A.HorizontalFlip(p=0.5),
        A.Affine(translate_percent=(-0.1, 0.1), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']),
        ToTensorV2(),
    ])
    eval_transform = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=preprocessing['interpolation']),
        A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']),
        ToTensorV2(),
    ])
    return train_transform, eval_transform

class PetDataset(Dataset):
    def __init__(self, frame, transform, with_labels=True):
        self.paths = frame.path.tolist()
        self.targets = frame[LABEL_COLS].to_numpy(dtype=np.float32) if with_labels else None
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            pixels = np.array(image.convert('RGB'))
        x = self.transform(image=pixels)['image']
        if self.targets is None:
            return x
        return x, torch.from_numpy(self.targets[index])

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(frame, training=False, with_labels=True):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        PetDataset(frame, train_transform if training else eval_transform, with_labels),
        batch_size=BATCH_SIZE, shuffle=training, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', drop_last=False,
        worker_init_fn=seed_worker, generator=generator,
    )

In [8]:
def metric_report(targets, probabilities, threshold=THRESHOLD):
    predictions = (probabilities >= threshold).astype(np.int32)
    rows = []
    for j, label in enumerate(LABEL_COLS):
        y = targets[:, j]
        auc = roc_auc_score(y, probabilities[:, j]) if np.unique(y).size == 2 else np.nan
        rows.append({
            'label': label,
            'positives': int(y.sum()),
            'predicted positives': int(predictions[:, j].sum()),
            'f1': f1_score(y, predictions[:, j], zero_division=0),
            'precision': precision_score(y, predictions[:, j], zero_division=0),
            'recall': recall_score(y, predictions[:, j], zero_division=0),
            'roc_auc': auc,
        })
    table = pd.DataFrame(rows).set_index('label')
    valid_auc = table.roc_auc.dropna()
    metrics = {
        'macro_f1': float(f1_score(targets, predictions, average='macro', zero_division=0)),
        'macro_roc_auc': float(valid_auc.mean()) if len(valid_auc) else float('nan'),
    }
    return metrics, table

def tune_thresholds(targets, probabilities):
    thresholds = np.full(len(LABEL_COLS), THRESHOLD, dtype=np.float64)
    candidates = np.round(np.linspace(0.05, 0.95, 91), 2)
    for j, label in enumerate(LABEL_COLS):
        if np.unique(targets[:, j]).size < 2:
            print(f'{label}: keeping {THRESHOLD}; validation does not contain both outcomes.')
            continue
        scores = np.array([
            f1_score(targets[:, j], probabilities[:, j] >= t, zero_division=0)
            for t in candidates
        ])
        tied = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
        best = tied[np.argmin(np.abs(candidates[tied] - THRESHOLD))]
        thresholds[j] = candidates[best]
    return thresholds

In [9]:

@torch.no_grad()
def update_ema():
    live = model.state_dict()
    for name, averaged in ema_model.state_dict().items():
        value = live[name].detach()
        if averaged.is_floating_point():
            averaged.mul_(EMA_DECAY).add_(value, alpha=1 - EMA_DECAY)
        else:
            averaged.copy_(value)

def train_one_epoch():
    global train_step
    model.train()
    total_loss, seen = 0.0, 0
    progress = tqdm(train_loader, desc='fine-tune', leave=False)
    for batch_index, (images, targets) in enumerate(progress):
        if batch_index % ACCUMULATION_STEPS == 0:
            optimizer.zero_grad(set_to_none=True)
            window_examples = min(BATCH_SIZE * ACCUMULATION_STEPS, len(train_loader.dataset) - seen)
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            loss = train_criterion(model(images), targets)
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite training loss: {loss.item()}')
        scaler.scale(loss * len(images) / window_examples).backward()
        total_loss += loss.item() * len(images)
        seen += len(images)
        end_window = (batch_index + 1) % ACCUMULATION_STEPS == 0 or batch_index + 1 == len(train_loader)
        if end_window:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                update_ema()
                scheduler.step()
                train_step += 1
            if wandb_run is not None and (train_step % WANDB_LOG_EVERY == 0 or batch_index + 1 == len(train_loader)):
                wandb_run.log({'train_step': train_step, 'batch/loss': loss.item(),
                               'batch/running_loss': total_loss / seen})
        progress.set_postfix(loss=f'{total_loss / seen:.5f}')
    return total_loss / seen

@torch.inference_mode()
def validate():
    model.eval()
    total_loss = 0.0
    all_targets, all_probabilities = [], []
    for images, targets in tqdm(val_loader, desc='validation', leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
        loss = val_criterion(logits.float(), targets)
        total_loss += loss.item() * len(images)
        all_targets.append(targets.cpu().numpy())
        all_probabilities.append(logits.float().sigmoid().cpu().numpy())
    targets = np.concatenate(all_targets)
    probabilities = np.concatenate(all_probabilities)
    assert np.isfinite(probabilities).all(), 'Non-finite validation probabilities.'
    metrics, table = metric_report(targets, probabilities)
    metrics['val_loss'] = total_loss / len(val_df)
    assert np.isfinite(metrics['val_loss']), 'Non-finite validation loss.'
    return metrics, table, probabilities

In [10]:
wandb_run = None
train_step = 0

def start_wandb_run(config):
    if not USE_WANDB or WANDB_MODE == 'disabled':
        return nullcontext(None)
    if WANDB_MODE not in {'online', 'offline'}:
        raise ValueError('WANDB_MODE must be online, offline, or disabled.')
    if WANDB_LOG_EVERY < 1:
        raise ValueError('WANDB_LOG_EVERY must be positive.')
    try:
        import wandb
    except ImportError:
        raise ImportError('Install wandb with %pip install -q wandb, or set USE_WANDB = False.') from None

    if WANDB_MODE == 'online' and not os.environ.get('WANDB_API_KEY'):
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception:
            raise RuntimeError(
                'Add the WANDB_API_KEY Kaggle Secret and grant notebook access, '
                'or set the WANDB_API_KEY environment variable. '
                'For local logging use WANDB_MODE = "offline".'
            ) from None
        if not api_key:
            raise RuntimeError('WANDB_API_KEY is empty. Check the Kaggle Secret.')
        os.environ['WANDB_API_KEY'] = api_key  # never print or put the key in run config

    id_path = OUTPUT_DIR / 'wandb_id.txt'
    run_id = id_path.read_text().strip() if id_path.is_file() else wandb.util.generate_id()
    id_path.write_text(run_id)
    return wandb.init(
        id=run_id, resume='allow', group=f'convnext-cv320-{cv_fingerprint[:8]}',
        project=WANDB_PROJECT, entity=WANDB_ENTITY, name=WANDB_RUN_NAME,
        mode=WANDB_MODE, config=config, dir=str(OUTPUT_DIR), save_code=False,
    )

In [11]:
@torch.inference_mode()
def predict_val_pair():
    model.eval()
    originals, averages = [], []
    for images, _ in tqdm(val_loader, desc=f'OOF fold {FOLD}: original + flip'):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
        original = logits.float().sigmoid()
        del logits
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits_flip = model(torch.flip(images, dims=[3]))
        originals.append(original.cpu().numpy())
        averages.append(((original + logits_flip.float().sigmoid()) * 0.5).cpu().numpy())
    return np.concatenate(originals), np.concatenate(averages)

def check_oof(path, held_out):
    result = pd.read_csv(path, dtype={'image_id': str})
    assert result.image_id.is_unique and set(result.image_id) == set(held_out.image_id)
    result = result.set_index('image_id').loc[held_out.image_id].reset_index()
    assert np.array_equal(result[LABEL_COLS].to_numpy(), held_out[LABEL_COLS].to_numpy())
    assert np.array_equal(result['fold'].to_numpy(), held_out['fold'].to_numpy())
    assert np.array_equal(result['cluster'].to_numpy(), held_out['cluster'].to_numpy())
    for suffix in ('_prob', '_prob_tta'):
        p = result[[label + suffix for label in LABEL_COLS]].to_numpy()
        assert p.shape == (len(held_out), len(LABEL_COLS)) and np.isfinite(p).all()
        assert ((p >= 0) & (p <= 1)).all()
    return result

## 4. Same-fold fine-tuning and EMA

Epoch 0 evaluates the source weights at the new resolution before any update. Each
later epoch evaluates raw and EMA weights with the unchanged validation function.
Select the best macro F1 at 0.5; lower validation loss breaks ties. The source-at-320
checkpoint is retained if no fine-tuned candidate is better. No validation images enter
training, and no fold is initialized from a different fold's fine-tuned weights.


In [12]:
for FOLD in FOLDS_TO_TRAIN:
    OUTPUT_DIR = CV_DIR / f'fold_{FOLD}'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_PATH, LAST_PATH = OUTPUT_DIR / 'best.pt', OUTPUT_DIR / 'last.pt'
    complete_path, oof_path = OUTPUT_DIR / 'complete.json', OUTPUT_DIR / 'oof.csv'
    train_df = frame.loc[frame.fold != FOLD].reset_index(drop=True)
    val_df = frame.loc[frame.fold == FOLD].reset_index(drop=True)
    assert len(train_df) and len(val_df) and set(train_df.cluster).isdisjoint(val_df.cluster)
    if complete_path.is_file():
        done = json.loads(complete_path.read_text())
        assert done['cv_fingerprint'] == cv_fingerprint and done['fold'] == FOLD
        assert done['checkpoint_sha256'] == file_digest(BEST_PATH)
        check_oof(oof_path, val_df)
        print(f'Fold {FOLD}: completed; skipping.')
        continue

    source_path = SOURCE_CV_DIR / f'fold_{FOLD}' / 'best.pt'
    assert file_digest(source_path) == source_manifest['checkpoint_hashes'][str(FOLD)]
    # Prove that the matching source model held out the same images.
    check_oof(SOURCE_CV_DIR / f'fold_{FOLD}' / 'oof.csv', val_df)
    source_checkpoint = torch.load(source_path, map_location='cpu', weights_only=True)
    cfg = source_checkpoint['config']
    assert cfg['fold'] == FOLD and cfg['cv_fingerprint'] == source_manifest['cv_fingerprint']
    assert cfg['model_name'] == MODEL_NAME and cfg['label_cols'] == LABEL_COLS
    assert cfg['preprocessing'] == source_manifest['preprocessing']
    seed_everything(SEED + FOLD)
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(LABEL_COLS)).to(DEVICE)
    model.load_state_dict(source_checkpoint['model'], strict=True)
    source_epoch = source_checkpoint['epoch']
    del source_checkpoint
    ema_model = copy.deepcopy(model).eval().requires_grad_(False)
    train_transform, eval_transform = make_transforms()
    train_loader, val_loader = make_loader(train_df, training=True), make_loader(val_df)
    positive_counts = train_df[LABEL_COLS].sum().to_numpy(dtype=np.float32)
    negative_counts = len(train_df) - positive_counts
    assert (positive_counts > 0).all() and (negative_counts > 0).all()
    pos_weight = torch.as_tensor(np.sqrt(negative_counts / positive_counts), device=DEVICE)
    train_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    val_criterion = nn.BCEWithLogitsLoss()
    classifier = list(model.get_classifier().parameters())
    head_ids = {id(parameter) for parameter in classifier}
    backbone = [parameter for parameter in model.parameters() if id(parameter) not in head_ids]
    assert classifier and backbone
    optimizer = torch.optim.AdamW([{'params': backbone, 'lr': LR}, {'params': classifier, 'lr': HEAD_LR}], weight_decay=WEIGHT_DECAY)
    updates_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
    total_updates = EPOCHS * updates_per_epoch
    warmup_updates = WARMUP_EPOCHS * updates_per_epoch
    def lr_factor(update):
        if update < warmup_updates:
            return 0.1 + 0.9 * update / max(warmup_updates, 1)
        progress = min(1.0, (update - warmup_updates) / max(total_updates - warmup_updates, 1))
        minimum = MIN_LR / LR
        return minimum + 0.5 * (1 - minimum) * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    data_config = timm.data.resolve_data_config({}, model=model)
    data_config['input_size'] = (3, IMAGE_SIZE, IMAGE_SIZE)
    fold_config = {**cv_config, 'fold': FOLD, 'num_classes': len(LABEL_COLS),
                   'image_size': IMAGE_SIZE, 'folds_path': str(FOLDS_PATH), 'data_config': data_config,
                   'source_checkpoint': str(source_path), 'source_epoch': source_epoch,
                   'pos_weight': pos_weight.cpu().tolist()}
    WANDB_RUN_NAME = f'convnext-cv320-fold{FOLD}'
    wandb_run, train_step = None, 0
    history, start_epoch, patience, training_complete = [], 1, 0, False
    fresh = not LAST_PATH.is_file()
    if not fresh:
        last = torch.load(LAST_PATH, map_location='cpu', weights_only=True)
        assert last['config']['cv_fingerprint'] == cv_fingerprint and last['config']['fold'] == FOLD
        assert BEST_PATH.is_file()
        model.load_state_dict(last['model'])
        ema_model.load_state_dict(last['ema'])
        optimizer.load_state_dict(last['optimizer'])
        scheduler.load_state_dict(last['scheduler'])
        scaler.load_state_dict(last['scaler'])
        history, start_epoch = last['history'], last['epoch'] + 1
        best_f1, best_loss, patience = last['best_f1'], last['best_loss'], last['patience']
        train_step, training_complete = last['train_step'], last['training_complete']
        initial_metrics = last['initial_metrics']
        print(f'Fold {FOLD}: resuming after fine-tuning epoch {last["epoch"]}.')
        del last
    else:
        if BEST_PATH.is_file():
            orphan = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
            assert orphan['epoch'] == 0 and orphan['config']['cv_fingerprint'] == cv_fingerprint, 'Restore last.pt or use a fresh CV_DIR.'
            del orphan
        initial_metrics, _, initial_probabilities = validate()
        val_targets = val_df[LABEL_COLS].to_numpy(dtype=np.int32)
        initial_metrics['val_accuracy'] = float(((initial_probabilities >= THRESHOLD) == val_targets).all(axis=1).mean())
        best_f1, best_loss = initial_metrics['macro_f1'], initial_metrics['val_loss']
        save_torch({'model': model.state_dict(), 'epoch': 0, 'metrics': initial_metrics,
                    'variant': 'source_at_320', 'config': fold_config}, BEST_PATH)
        # Initial last.pt permits recovery even if the first epoch is interrupted.
        save_torch({'model': model.state_dict(), 'ema': ema_model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'epoch': 0,
                    'history': history, 'best_f1': best_f1, 'best_loss': best_loss, 'patience': patience,
                    'train_step': train_step, 'training_complete': False, 'initial_metrics': initial_metrics,
                    'config': fold_config}, LAST_PATH)
        print(f'Fold {FOLD}: source weights at 320, F1={best_f1:.5f} before fine-tuning.')
    try:
        if not training_complete:
            with start_wandb_run(fold_config) as wandb_run:
                if wandb_run is not None:
                    wandb_run.define_metric('train_step')
                    wandb_run.define_metric('batch/*', step_metric='train_step')
                    wandb_run.define_metric('epoch')
                    wandb_run.define_metric('epoch_metrics/*', step_metric='epoch')
                    for metric in ('val_f1', 'val_loss', 'train_loss_epoch', 'lr'):
                        wandb_run.define_metric(metric, step_metric='epoch')
                    print('W&B:', wandb_run.url if WANDB_MODE == 'online' else wandb_run.dir)
                    if fresh:
                        wandb_run.log({'epoch': 0, 'val_f1': initial_metrics['macro_f1'],
                                       'val_loss': initial_metrics['val_loss'],
                                       'epoch_metrics/val_f1': initial_metrics['macro_f1']})
                for epoch in range(start_epoch, EPOCHS + 1):
                    epoch_seed = SEED + FOLD * 1000 + epoch
                    seed_everything(epoch_seed)
                    if hasattr(train_transform, 'set_random_seed'):
                        train_transform.set_random_seed(epoch_seed)
                    train_loader.generator.manual_seed(epoch_seed)
                    started = perf_counter()
                    epoch_lr, head_lr = [group['lr'] for group in optimizer.param_groups]
                    train_loss = train_one_epoch()
                    raw_model = model
                    raw_metrics, raw_table, raw_probabilities = validate()
                    try:
                        model = ema_model
                        ema_metrics, ema_table, ema_probabilities = validate()
                    finally:
                        model = raw_model
                    ema_wins = (ema_metrics['macro_f1'], -ema_metrics['val_loss']) > (raw_metrics['macro_f1'], -raw_metrics['val_loss'])
                    metrics, table, probabilities = (ema_metrics, ema_table, ema_probabilities) if ema_wins else (raw_metrics, raw_table, raw_probabilities)
                    chosen = ema_model if ema_wins else model
                    variant = 'ema' if ema_wins else 'raw'
                    targets = val_df[LABEL_COLS].to_numpy(dtype=np.int32)
                    metrics['val_accuracy'] = float(((probabilities >= THRESHOLD) == targets).all(axis=1).mean())
                    improved = metrics['macro_f1'] > best_f1
                    patience = 0 if improved else patience + 1
                    if improved or (metrics['macro_f1'] == best_f1 and metrics['val_loss'] < best_loss):
                        best_f1, best_loss = metrics['macro_f1'], metrics['val_loss']
                        save_torch({'model': chosen.state_dict(), 'epoch': epoch, 'metrics': metrics,
                                    'variant': variant, 'config': fold_config}, BEST_PATH)
                    row = {'epoch': epoch, 'train_loss': train_loss, **metrics, 'lr': epoch_lr, 'head_lr': head_lr,
                           'val_f1_raw': raw_metrics['macro_f1'], 'val_f1_ema': ema_metrics['macro_f1'],
                           'epoch_seconds': perf_counter() - started, 'epochs_without_improvement': patience}
                    history.append(row)
                    training_complete = epoch == EPOCHS or patience >= EARLY_STOPPING_PATIENCE
                    save_torch({'model': model.state_dict(), 'ema': ema_model.state_dict(), 'optimizer': optimizer.state_dict(),
                                'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(), 'epoch': epoch,
                                'history': history, 'best_f1': best_f1, 'best_loss': best_loss, 'patience': patience,
                                'train_step': train_step, 'training_complete': training_complete,
                                'initial_metrics': initial_metrics, 'config': fold_config}, LAST_PATH)
                    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'history.csv', index=False)
                    print(f'Fold {FOLD} | epoch {epoch}/{EPOCHS} | raw F1 {raw_metrics["macro_f1"]:.5f} | EMA F1 {ema_metrics["macro_f1"]:.5f} | best {best_f1:.5f} | patience {patience}/{EARLY_STOPPING_PATIENCE}')
                    if wandb_run is not None:
                        logs = {'epoch': epoch, 'val_f1': metrics['macro_f1'], 'val_loss': metrics['val_loss'],
                                'train_loss_epoch': train_loss, 'lr': epoch_lr}
                        logs.update({f'epoch_metrics/{key}': value for key, value in row.items() if key != 'epoch' and np.isfinite(value)})
                        logs['epoch_metrics/val_f1'] = metrics['macro_f1']
                        logs.update({f'epoch_metrics/f1_{label}': float(table.loc[label, 'f1']) for label in LABEL_COLS})
                        wandb_run.log(logs)
                        wandb_run.summary.update({'best/macro_f1': best_f1, 'completed_epochs': epoch})
                    if training_complete:
                        break
    finally:
        wandb_run = None
    best_checkpoint = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
    model.load_state_dict(best_checkpoint['model'])
    originals, averages = predict_val_pair()
    result = val_df[['image_id', 'cluster', 'fold', *LABEL_COLS]].copy()
    for j, label in enumerate(LABEL_COLS):
        result[label + '_prob'] = originals[:, j]
        result[label + '_prob_tta'] = averages[:, j]
    result.to_csv(oof_path, index=False)
    check_oof(oof_path, val_df)
    source_rows = source_oof.set_index('image_id').loc[val_df.image_id]
    source_p = source_rows[[label + '_prob' for label in LABEL_COLS]].to_numpy()
    source_224_f1 = f1_score(val_df[LABEL_COLS].to_numpy(), source_p >= THRESHOLD, average='macro', zero_division=0)
    comparison = {'fold': FOLD, 'source_224_f1_at_0_5': float(source_224_f1),
                  'source_at_320_f1_at_0_5': initial_metrics['macro_f1'],
                  'best_320_f1_at_0_5': best_checkpoint['metrics']['macro_f1'],
                  'best_epoch': best_checkpoint['epoch'], 'best_variant': best_checkpoint['variant']}
    save_json(comparison, OUTPUT_DIR / 'comparison.json')
    save_json({'fold': FOLD, 'cv_fingerprint': cv_fingerprint, 'checkpoint_sha256': file_digest(BEST_PATH),
               'best_epoch': best_checkpoint['epoch'], 'validation_images': len(val_df)}, complete_path)
    print('Completed:', comparison)
    for name in ('model', 'ema_model', 'raw_model', 'chosen', 'classifier', 'backbone', 'optimizer', 'scheduler', 'scaler', 'best_checkpoint'):
        globals().pop(name, None)
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

Fold 0: source weights at 320, F1=0.92439 before fine-tuning.


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mkovalenko (mkovalenko-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B: https://wandb.ai/yrachynska-kyiv-school-of-economics/cats-and-dogs/runs/be7fzl5s


Fold 0 | epoch 1/6 | raw F1 0.95935 | EMA F1 0.95924 | best 0.95935 | patience 0/4


Fold 0 | epoch 2/6 | raw F1 0.95915 | EMA F1 0.95905 | best 0.95935 | patience 1/4


...

Fold 4 | epoch 5/6 | raw F1 0.95337 | EMA F1 0.95337 | best 0.96212 | patience 2/4


Fold 4 | epoch 6/6 | raw F1 0.95337 | EMA F1 0.95337 | best 0.96212 | patience 3/4


batch/loss,▁▂▁▁▁▂▂▁▁▁█▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/running_loss,▇█▆▅▅▅▄▁▁▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▂▃▅▆▇█
epoch_metrics/epoch_seconds,█▁▂▁▂▅
epoch_metrics/epochs_without_improvement,▁▁▁▃▆█
epoch_metrics/f1_Biatrix,▁▁▁▁▁▁
epoch_metrics/f1_Cat,▁▅▅▁██
epoch_metrics/f1_Dog,▁▁██▁▁
epoch_metrics/f1_Moris,▁▁██▁▁
epoch_metrics/f1_Motya,▁▁▁▁▁▁
+15,...


OOF fold 4: original + flip: 100%|██████████| 154/154 [00:28<00:00,  5.44it/s]


Completed: {'fold': 4, 'source_224_f1_at_0_5': 0.9670232124992101, 'source_at_320_f1_at_0_5': 0.9401950413097797, 'best_320_f1_at_0_5': 0.9621247320021169, 'best_epoch': 3, 'best_variant': 'ema'}


## 5. Full OOF calibration

Require all five completed folds, exactly one held-out prediction per training image.
Do not average predictions of other fold models on that image: those models trained on it.
Thresholds are fitted separately for original and TTA OOF probabilities. Test inference
will average the five models' probabilities and use these OOF-derived thresholds. The
single-model OOF and five-model test probability distributions may differ; evaluate the
result on Kaggle. This notebook does not submit anything automatically.

The chosen 320 weights may be raw, EMA, or epoch-0 source weights. All OOF probabilities are recomputed with the same saved 320 evaluation preprocessing.

In [13]:
missing_folds = [f for f in range(5) if not (CV_DIR / f'fold_{f}' / 'complete.json').is_file()]
if missing_folds:
    print('OOF calibration not ready. Train/import remaining folds:', missing_folds)
else:
    parts = []
    checkpoint_hashes = {}
    for f in range(5):
        folder = CV_DIR / f'fold_{f}'
        done = json.loads((folder / 'complete.json').read_text())
        assert done['cv_fingerprint'] == cv_fingerprint and done['fold'] == f
        assert done['checkpoint_sha256'] == file_digest(folder / 'best.pt')
        checkpoint_hashes[str(f)] = done['checkpoint_sha256']
        held_out = frame.loc[frame.fold == f].reset_index(drop=True)
        parts.append(check_oof(folder / 'oof.csv', held_out))
    oof = pd.concat(parts, ignore_index=True)
    assert oof.image_id.is_unique and set(oof.image_id) == set(frame.image_id)
    oof = oof.set_index('image_id').loc[frame.image_id].reset_index()
    targets = frame[LABEL_COLS].to_numpy(dtype=np.int32)
    all_thresholds, reports = {}, []
    for method, suffix in [('original', '_prob'), ('tta', '_prob_tta')]:
        probabilities = oof[[label + suffix for label in LABEL_COLS]].to_numpy()
        thresholds = tune_thresholds(targets, probabilities)
        all_thresholds[method] = thresholds.tolist()
        for label, cutoffs in [('fixed_0_5', 0.5), ('fitted_oof', thresholds)]:
            report, _ = metric_report(targets, probabilities, cutoffs)
            report['val_accuracy'] = float(((probabilities >= cutoffs) == targets).all(axis=1).mean())
            reports.append({'method': method, 'threshold_source': label, **report})
    summary = pd.DataFrame(reports)
    display(summary)
    oof.to_csv(CV_DIR / 'oof_predictions.csv', index=False)
    summary.to_csv(CV_DIR / 'oof_metrics.csv', index=False)
    save_json({'model_name': MODEL_NAME, 'label_cols': LABEL_COLS, 'folds': list(range(5)),
               'cv_fingerprint': cv_fingerprint, 'folds_fingerprint': folds_fingerprint,
               'preprocessing': preprocessing, 'thresholds': all_thresholds,
               'checkpoint_hashes': checkpoint_hashes,
               'aggregation': 'equal_mean_sigmoid_probabilities', 'tta': 'original_and_horizontal_flip',
               'threshold_source': 'all_held_out_oof_predictions'}, CV_DIR / 'ensemble_config.json')
    print('Ready: run cnn_2_ensemble_320.ipynb with CV_DIR =', CV_DIR)

comparisons = [json.loads((CV_DIR / f'fold_{f}' / 'comparison.json').read_text())
               for f in range(5) if (CV_DIR / f'fold_{f}' / 'comparison.json').is_file()]
if comparisons:
    display(pd.DataFrame(comparisons))


,method,threshold_source,macro_f1,macro_roc_auc,val_accuracy
0,original,fixed_0_5,0.976614,0.999723,0.993230
1,original,fitted_oof,0.978687,0.999723,0.993148
2,tta,fixed_0_5,0.974270,0.999788,0.993393
3,tta,fitted_oof,0.978614,0.999788,0.993638


Ready: run cnn_2_ensemble_320.ipynb with CV_DIR = /kaggle/working/convnext_cv_320


,fold,source_224_f1_at_0_5,source_at_320_f1_at_0_5,best_320_f1_at_0_5,best_epoch,best_variant
0,0,0.972387,0.924389,0.972686,5,ema
1,1,0.974429,0.910690,0.981466,1,ema
2,2,0.989132,0.905706,0.990119,2,ema
3,3,0.989916,0.953036,0.990020,3,raw
4,4,0.967023,0.940195,0.962125,3,ema
